# EpiAI 一站式教程
从原始数据到部署预测 + 可视化 + 预警评估

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path

from EpiAI import ForecastTask
from EpiAI.dataset import TimeSplit
from EpiAI.viz import plot_test_predictions, plot_forecast
from EpiAI.evaluate import warning_report

## 1. 加载数据

In [ ]:
DATA_PATH = Path.home() / "infective_disease" / "pingtai" / "merged_clean.csv"
DISEASE = "登革热 Dengue fever"

df_raw = pd.read_csv(DATA_PATH)
df_raw["Year/Month"] = pd.to_datetime(df_raw["Year/Month"])
df = (
    df_raw.query("Diseases == @DISEASE")
    [["Year/Month", "Case number"]]
    .rename(columns={"Year/Month": "time", "Case number": "cases"})
    .reset_index(drop=True)
)
df = df[df["time"].dt.year >= 2010].reset_index(drop=True)
print(f"数据: {len(df)} 行, {df['time'].min().date()} → {df['time'].max().date()}")

## 2. 定义预测任务

In [ ]:
task = ForecastTask(
    data=df,
    target="cases",
    time_col="time",
    freq="MS",
    horizon=12,
    split=TimeSplit(train_end="2022-05-01", val_end="2023-05-01"),
)
print(f"Task: {task}")
print(f"季节周期: {task.freq_to_season_length()} 期/年")

## 3. 批量训练 + 排行榜

In [ ]:
vault = task.compare([
    "sf_holt_winters",
    "prophet",
    "arima",
    "sf_mstl",
], cache_dir="./cache")
print("\n排行榜（按 R² 排序）:")
display(vault.summary())

## 4. 测试集预测对比图

真实值（黑线） vs 各模型在测试集上的预测值。

In [ ]:
plot_test_predictions(
    bundle=task.build_bundle(),
    vault=vault,
    title=f"{DISEASE} 测试集预测对比",
)

## 5. 预警评估 — 选择最佳预警方法

利用 `compare()` 已有的测试集预测结果，零成本评估多种预警阈值方法，
指导 deploy 时选择哪个风险算法。

In [20]:
# 对比多种阈值方法
rows = []
for method, level, desc in [
    ('quantile', 0.80, 'p80'),
    # ('quantile', 0.90, 'p90'),
    # ('quantile', 0.95, 'p95'),
    # ('zscore',   0.5,  '0.5σ'),
    ('zscore',   1.0,  '1.0σ'),
]:
    r = warning_report(vault, vault.bundle, task, method=method, level=level)
    for _, row in r.iterrows():
        rows.append({
            '方法': desc,
            '模型': row['model'],
            'AUC': f"{row['auc']:.3f}",
            '命中率': f"{row['hit_rate']:.2f}",
            '虚报率': f"{row['fa_rate']:.2f}",
            'F1': f"{row['f1']:.2f}",
        })

display(pd.DataFrame(rows))

,方法,模型,AUC,命中率,虚报率,F1
0,p80,sf_mstl,0.988,0.94,0.00,0.97
1,p80,linearreg,0.956,0.94,0.45,0.75
2,p80,mlp,0.953,0.88,0.15,0.85
3,p80,sf_auto_ces,0.906,0.75,0.00,0.86
4,p80,sf_naive,0.500,0.00,0.00,nan
5,p80,sf_tsb,0.500,0.00,0.00,nan
6,1.0σ,sf_auto_ces,0.980,0.00,0.00,nan
7,1.0σ,sf_mstl,0.966,0.00,0.00,nan
8,1.0σ,linearreg,0.946,0.00,0.00,nan
9,1.0σ,mlp,0.887,0.57,0.07,0.62


### 解读

| 方法 | 含义 |
|------|------|
| `p80/p90/p95` | 预测值超过历史同期 80/90/95% 分位 → 预警 |
| `0.5σ/1.0σ` | 预测值超过历史均值 + 0.5/1.0 标准差 → 预警 |
| `th500` | 预测值 > 500 → 预警 |

**AUC**：预测值的排序能力（不受阈值影响），越高越好
**命中率/虚报率**：在具体阈值下的实战表现

选 AUC 最高 + 虚报率可接受的组合用于 deploy。

## 6. 部署 + 未来预测

In [ ]:
best_model = vault.best("R2")
print(f"最佳模型: {best_model}")

runtime = task.deploy(best_model, vault, risk="quantile")
print(f"Runtime: {runtime}")

# 未来预测图
hist = runtime.data_table.tail(20)
pred = runtime.predict(horizon=12, level=95)

plot_forecast(
    hist_df=hist,
    pred_df=pred,
    metrics_df=vault.metrics,
    title=f"{DISEASE} 未来 12 期预测",
)